# CC-GAVN: a 4.76M candidate-conditioned searchless chess model

Objective: test whether allowing the candidate action to participate in every geometric-attention layer improves action-value distillation over the 5.30M legacy GAVN, while remaining below a strict 5M-parameter budget.

Success criterion: choose a configuration only by held-out ChessBench development loss, then perform one frozen evaluation with the corrected distribution-expectation engine on all MATE subsets and the official puzzle protocol. No frozen test row is used for training or model selection.


In [ ]:
from pathlib import Path
import os, subprocess, sys, time

REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
SL_REPO = Path('/kaggle/working/searchless_chess')
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
assert (REPO / 'scripts/train_ccgavn.py').exists()

# The write token must come from Kaggle Secrets or a private dataset attachment.
# There is deliberately no hard-coded credential fallback.
for _ in range(18):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
        if token and token.strip():
            os.environ['HF_WRITE_TOKEN'] = token.strip()
            break
    except Exception as exc:
        print(f'HF secret retry in 10s: {exc}')
        time.sleep(10)
if not os.environ.get('HF_WRITE_TOKEN'):
    import glob
    for credential_file in sorted(glob.glob('/kaggle/input/*/hf_token.txt')):
        token = Path(credential_file).read_text().strip()
        if token:
            os.environ['HF_WRITE_TOKEN'] = token
            print('HF credential loaded from private dataset attachment')
            break
if not os.environ.get('HF_WRITE_TOKEN'):
    raise RuntimeError('HF_WRITE_TOKEN is required: checkpoint persistence is mandatory')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'python-chess'], check=True)
os.chdir(REPO)


## Experimental plan

- Architecture: CC-GAVN v1 uses 64 square tokens and one candidate-action token. The candidate token attends to and is updated by the board at every layer.
- Inductive bias: corrected live geometric relations (knight, king adjacency, rank, file, diagonal, other) plus an exact horizontal-reflection augmentation that remaps FEN rule state and UCI action together.
- Training target: 128-bin action-return distribution from the existing teacher plus the original Stockfish return bucket. The distribution expectation is the only canonical decision score.
- Selection: reserve a deterministic position-level 1% development fold from ChessBench. It is excluded from updates; no MATE or puzzle metric selects a model.
- Controls: same data, optimization budget, seed count, and parameter accounting as the repaired GAVN baseline.


In [ ]:
# Mandatory end-to-end smoke gate before an expensive run.
# max-records restricts the run to the first shard and keeps the production
# trainer's Hugging Face persistence, resume, split, and checkpoint paths live.
smoke = Path('/kaggle/working/ccgavn-smoke')
smoke_cmd = [
    sys.executable, 'scripts/train_ccgavn.py',
    '--hf-shards', 'chessbench-full-build', '--outdir', str(smoke),
    '--sl-repo', str(SL_REPO), '--dim', '96', '--layers', '2', '--heads', '4',
    '--batch', '32', '--steps', '20', '--max-records', '4096',
    '--dev-mod', '20', '--dev-batch', '128', '--ckpt-every', '20',
    '--hf-run', 'smoke-ccgavn-disposable', '--hf-upload-every', '1',
]
subprocess.run(smoke_cmd, check=True)
print('CC-GAVN persistence and training smoke test passed')


In [ ]:
# Production configuration: strict 5M budget (actual count: 4,762,088).
RUN_ID = 'ccgavn-5m-seed0'
DIM, LAYERS, HEADS = 208, 8, 8
SEED, STEPS = 0, 160_000
OUT = Path('/kaggle/working') / RUN_ID
production_cmd = [
    sys.executable, 'scripts/train_ccgavn.py',
    '--hf-shards', 'chessbench-full-build', '--outdir', str(OUT),
    '--sl-repo', str(SL_REPO), '--dim', str(DIM), '--layers', str(LAYERS),
    '--heads', str(HEADS), '--batch', '2048', '--steps', str(STEPS),
    '--lr', '0.0005', '--warmup', '2000', '--temperature', '1.0',
    '--w-dist', '1.0', '--w-ce', '0.25', '--reflect-prob', '0.5',
    '--dev-mod', '100', '--dev-fold', '0', '--dev-batch', '8192',
    '--seed', str(SEED), '--ckpt-every', '5000',
    '--hf-repo', 'vedangfake/chess-slm-benchmark', '--hf-run', RUN_ID,
    '--hf-upload-every', '1800', '--resume-from-hf',
]
print(' '.join(production_cmd))
subprocess.run(production_cmd, check=True)


## Evaluation protocol

After selecting the checkpoint by development loss, run `scripts/eval_gavn.py` in a separate CPU evaluation job. Use `--score auto` (which resolves to the trained distribution expectation), all four frozen MATE subsets, and the official puzzle CSV. Report every seed, parameter count, latency, confidence intervals, and the legacy 5M GAVN control. Do not make a frontier claim from the smoke run or the development split.
